# NC-SRAG P1+P3: multi-family large-model runs

**Runtime:** Colab Pro+ **A100**. Runtime > Change runtime type > A100, then Run all.

Runs the full 11-condition controlled grid on several instruct models (five
families, 1.5B to 9B) with true token-logprob and entropy features plus
bidirectional-NLI semantic entropy. Each model's GPU memory is freed before the
next loads, the NLI scorer is loaded once, and prompts are built to work across
all chat templates. Results checkpoint to Google Drive and resume on
disconnect.

In [ ]:
# 1. install + mount Drive
!pip -q install transformers accelerate rank_bm25 scikit-learn scipy sentencepiece
from google.colab import drive; drive.mount('/content/drive')
import os; os.makedirs('data', exist_ok=True)
# Gated models (Gemma, Llama) need a token: add a Colab secret named HF_TOKEN
# (key icon, left sidebar) and accept each model's license on huggingface.co.

In [ ]:
# 2. upload the corpus: select docs.json AND questions.json from pilot_v2/data
from google.colab import files
up = files.upload()
for fn in up: os.rename(fn, f'data/{fn}')
print(sorted(os.listdir('data')))

In [ ]:
# 3. sanity check
assert os.path.exists('data/docs.json') and os.path.exists('data/questions.json'), \
    'Upload docs.json and questions.json into data/ in the cell above.'
print('corpus OK')

In [ ]:
%%writefile harness.py
import os, json, re, math, time, random, gc
import numpy as np, torch
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
WORK="/content/drive/MyDrive/ncsrag"; os.makedirs(WORK, exist_ok=True)

docs=json.load(open("data/docs.json")); questions=json.load(open("data/questions.json"))
bm25=BM25Okapi([re.findall(r"\w+",d["text"].lower()) for d in docs])
by_kind={}; gold_idx={}; contra_idx={}; distract_idx={}
for i,d in enumerate(docs):
    by_kind.setdefault(d["kind"],[]).append(i); k=(d.get("entity"),d.get("attr"))
    if d["kind"]=="gold": gold_idx[k]=i
    elif d["kind"]=="contra": contra_idx[k]=i
    elif d["kind"]=="distractor": distract_idx[(d.get("target"),d.get("attr"))]=i

# --- NLI semantic-entropy scorer: loaded ONCE, float32 (DeBERTa needs fp32) ---
NLI="microsoft/deberta-large-mnli"
nli_tok=AutoTokenizer.from_pretrained(NLI)
nli=AutoModelForSequenceClassification.from_pretrained(NLI,torch_dtype=torch.float32).to("cuda").eval()
@torch.no_grad()
def entails(a,b):
    x=nli_tok(a,b,return_tensors="pt",truncation=True,max_length=128).to(nli.device)
    return nli(**x).logits.argmax(-1).item()==2   # 2 = entailment (MNLI)
def semantic_entropy(samples):
    clusters=[]
    for s in samples:
        placed=False
        for c in clusters:
            if entails(s,c[0]) and entails(c[0],s): c.append(s); placed=True; break
        if not placed: clusters.append([s])
    tot=len(samples)
    return float(-sum((len(c)/tot)*math.log(len(c)/tot) for c in clusters))

def retrieve(q,k=4):
    sc=bm25.get_scores(re.findall(r"\w+",q.lower()))
    return [int(i) for i in np.argsort(-sc) if docs[i]["kind"] not in ("contra","distractor")][:k], sc
def build(q,cond,rng):
    if cond=="closedbook": return [],None
    base,sc=retrieve(q["question"]); gi=gold_idx[(q["entity"],q["attr"])]
    if gi not in base: base=[gi]+base[:3]
    ci=contra_idx.get((q["entity"],q["attr"]))
    if cond=="clean": ctx=list(base); rng.shuffle(ctx)
    elif cond.startswith("irr"):
        n={"irr25":1,"irr50":2,"irr75":3,"irr100":4}[cond]
        ctx=(base[:4-n] if cond!="irr100" else [])+rng.sample(by_kind["irrelevant"],n); rng.shuffle(ctx)
    elif cond=="contra_r1": ctx=[ci]+[d for d in base if d!=ci][:3]
    elif cond=="contra_r4": ctx=[d for d in base if d!=ci][:3]+[ci]
    elif cond=="contra_only": ctx=[ci]+rng.sample(by_kind["irrelevant"],3); rng.shuffle(ctx)
    elif cond=="distractor":
        di=distract_idx.get((q["entity"],q["attr"]))
        ctx=([di]+[d for d in base if d!=di][:3]) if di is not None else [base[0],ci]+[d for d in base[1:] if d!=ci][:2]
    elif cond=="mixed": ctx=[ci]+[d for d in base if d!=ci][:2]+rng.sample(by_kind["irrelevant"],1); rng.shuffle(ctx)
    return ctx,sc
def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
ABST=re.compile(r"(don.?t know|unknown|cannot|not (mentioned|stated|provided|given)|no answer)")
def label(a,g):
    a,g=norm(a),norm(g)
    if not a or ABST.search(a): return "abstain"
    if g in a or (a in g and len(a)>2): return "correct"
    if re.fullmatch(r"\d{4}",g) and g in re.findall(r"\d{4}",a): return "correct"
    return "hallucination"

def make_prompt(tok, sysmsg, user):
    # robust across families: some templates (Gemma, some Mistral) reject a system role
    try:
        return tok.apply_chat_template([{"role":"system","content":sysmsg},
                                        {"role":"user","content":user}],
                                       tokenize=False, add_generation_prompt=True)
    except Exception:
        return tok.apply_chat_template([{"role":"user","content":sysmsg+"\n\n"+user}],
                                       tokenize=False, add_generation_prompt=True)

CONDS=["closedbook","clean","irr25","irr50","irr75","irr100",
       "contra_r1","contra_r4","contra_only","distractor","mixed"]
SYS="Answer with the shortest factual answer. If the context lacks the answer, answer exactly: unknown"

def run(model_id):
    TAG=re.sub(r"[^a-zA-Z0-9]+","_",model_id.split("/")[-1]).lower()
    NSAMP=int(os.environ.get("NSAMP","5"))
    tok=AutoTokenizer.from_pretrained(model_id)
    model=AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=torch.bfloat16,device_map="auto").eval()

    @torch.no_grad()
    def gen(p, sample=False, n=1):
        enc=tok(p,return_tensors="pt",truncation=True,max_length=1024).to(model.device)
        out=model.generate(**enc,max_new_tokens=16,do_sample=sample,
            temperature=0.7 if sample else None, top_p=0.9 if sample else None,
            num_return_sequences=n, output_scores=True, return_dict_in_generate=True,
            pad_token_id=tok.eos_token_id)
        seqs=out.sequences[:,enc["input_ids"].shape[1]:]
        texts=[tok.decode(s,skip_special_tokens=True).strip() for s in seqs]
        lps=[]; ents=[]
        for step,sc in enumerate(out.scores):
            lp=torch.log_softmax(sc[0].float(),-1); tid=seqs[0][step] if step<seqs.shape[1] else seqs[0][-1]
            lps.append(float(lp[tid])); pr=lp.exp(); ents.append(float(-(pr*lp).sum()))
        return texts,(lps or [0.0]),(ents or [0.0])

    part=f"{WORK}/results_{TAG}_partial.json"
    rows=json.load(open(part)) if os.path.exists(part) else []
    done={(r["qid"],r["cond"]) for r in rows}; t0=time.time()
    for qi,q in enumerate(questions):
        rng=random.Random(1000+qi)
        for cond in CONDS:
            if (q["qid"],cond) in done: continue
            ctx,sc=build(q,cond,rng); p=make_prompt(tok,SYS,
                ("Context:\n"+"\n".join("- "+docs[i]["text"] for i in ctx)+"\n\n" if ctx else "")+f"Question: {q['question']}")
            texts,lps,ents=gen(p); g=texts[0]; lab=label(g,q["gold"])
            samples,_,_=gen(p,sample=True,n=NSAMP); se=semantic_entropy(samples)
            if ctx:
                cs=sorted([float(sc[i]) for i in ctx],reverse=True)
                feat=dict(ret_mean=float(np.mean(cs)),ret_top1=cs[0],ret_margin=cs[0]-cs[1],
                          ret_min=cs[-1],ret_std=float(np.std(cs)),has_ctx=1.0)
            else: feat=dict(ret_mean=0,ret_top1=0,ret_margin=0,ret_min=0,ret_std=0,has_ctx=0.0)
            feat.update(lp_mean=float(np.mean(lps)),lp_min=float(np.min(lps)),
                        ent_mean=float(np.mean(ents)),ent_max=float(np.max(ents)),
                        ent_first=ents[0],ans_len=len(lps),se=se)
            rows.append(dict(qid=q["qid"],regime=q["regime"],cond=cond,gold=q["gold"],answer=g,label=lab,**feat))
            done.add((q["qid"],cond))
        if qi%5==0:
            print(f"  {TAG}: q {qi}/{len(questions)} rows={len(rows)} t={time.time()-t0:.0f}s",flush=True)
            json.dump(rows,open(part,"w"))
    json.dump(rows,open(f"{WORK}/results_{TAG}.json","w"))
    print("DONE",TAG,len(rows),flush=True)
    # free GPU before the next model
    del model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 4. (optional) verify HF token + gated access before the long run
try:
    from huggingface_hub import whoami, auth_check
    print('HF user:', whoami().get('name','?'))
    for m in ["google/gemma-2-9b-it","meta-llama/Llama-3.1-8B-Instruct"]:
        try: auth_check(m); print('access OK ', m)
        except Exception as e: print('NO access', m, '->', type(e).__name__, '(it will be skipped)')
except Exception as e:
    print('No HF token set; gated models will be skipped. Open models still run. ->', type(e).__name__)

In [ ]:
# 5. run every model (memory freed between models; failures are skipped, not fatal)
import importlib, harness; importlib.reload(harness)
MODELS = ["microsoft/Phi-3.5-mini-instruct",
          "mistralai/Mistral-7B-Instruct-v0.3",
          "Qwen/Qwen2.5-1.5B-Instruct",
          "Qwen/Qwen2.5-3B-Instruct",
          "google/gemma-2-9b-it",
          "meta-llama/Llama-3.1-8B-Instruct"]
for m in MODELS:
    print('\n=== RUN', m, '===', flush=True)
    try:
        harness.run(m)
    except Exception as e:
        import gc, torch
        print('SKIPPED', m, '->', repr(e)[:300])
        gc.collect(); torch.cuda.empty_cache()
print('\nALL DONE. Files in MyDrive/ncsrag/')

## After the run
In `MyDrive/ncsrag/` you will have one `results_<tag>.json` per model that ran.
Download them into `pilot_v2/data/` on your PC and send them over; each is
analysed with `analyze_qwen7b.py <tag>` (retrieval/model/combined AUROC, the
semantic-entropy baseline, and conformal-under-shift), then folded into the
multi-family generality table. Expected tags: `phi_3_5_mini_instruct`,
`mistral_7b_instruct_v0_3`, `qwen2_5_1_5b_instruct`, `qwen2_5_3b_instruct`,
`gemma_2_9b_it`, `llama_3_1_8b_instruct`.